In [ ]:
import typesense
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

In [ ]:
import typesense

client = typesense.Client({
    'nodes': [{
        'host': 'p6ztcf5hybnxavmsp-1.a2.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': os.getenv('TYPESENSE_API_KEY'),
    'connection_timeout_seconds': 60
})

# Quick test to see if the key works
try:
    print("Testing connection...")
    print(client.collections.retrieve())
    print(" Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

In [ ]:
client

In [ ]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

In [ ]:
source_candidates = [Path('example.txt'), Path('Typesense/example.txt')]
source_path = next((path for path in source_candidates if path.exists()), None)
if source_path is None:
    raise FileNotFoundError('Could not find Typesense/example.txt')

loader = TextLoader(str(source_path), encoding='utf-8')
documents= loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

In [ ]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "p6ztcf5hybnxavmsp-1.a2.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": os.getenv('TYPESENSE_API_KEY'),
        "typesense_collection_name": "lang-chain"
    },
    
)

In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if not google_api_key:
    raise RuntimeError('GOOGLE_API_KEY is missing. Add it to the project .env file.')

llm = ChatGoogleGenerativeAI(
    model=os.getenv('GOOGLE_MODEL', 'gemini-2.5-flash'),
    temperature=0,
)

def ask_typesense(question: str) -> str:
    retrieved_docs = docsearch.similarity_search(question, k=4)
    context = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    prompt = f'''Answer the question using only the context below.
If the answer is not in the context, say that you do not know.

Context:
{context}

Question: {question}
Answer:'''
    response = llm.invoke(prompt)
    return response.content


In [ ]:
question = input('What is LSTM: ')
answer = ask_typesense(question)
print(answer)